# LifeMate — import teammate Qwen3.5 Thinking and compare agent variants

This notebook **does not train anything**. It imports your teammate's selected adapter, compares three inference agents on the original validation split, and runs public inference with the winner. All implementation comes from Git. Your earlier training notebooks remain unchanged.

**Pipeline:** trained Gipformer ASR → topical history segmentation → LLMLingua retrieval keys → BM25+BGE history selection → Qwen app router → tool-schema loading and tool execution → submission.

The model is the teammate's **Qwen3.5-4B Thinking LoRA**, with rank/alpha/base revisions read from the export (the supplied recipe uses rank 8 / alpha 16). It cannot be loaded into our previous compact-agent protocol. Only actual ToolEnv calls enter the submission; app routing and schema loading remain internal.

Your teammate reports **56.7 public**. The supplied notebook records **73.8763 on 82 clean-text held-out samples**, not a recorded public evaluation. This notebook does not claim an improved leaderboard score before running it.


## 1. Download these files from your teammate's Step 12

From `/kaggle/working/lifemate_qwen35_4b_thinking_lora_bm25_3ep_v1/`, download:

| File | Upload location under `MyDrive/Zalo-AI-Challenge-2026/` | Purpose |
|---|---|---|
| `best_adapter.zip` | `imports/teammate_topic_20261109/best_adapter.zip` | Required. Selected adapter, exact tokenizer, pinned model revisions and agent configuration. |
| `reports.zip` | `imports/teammate_topic_20261109/reports.zip` | Required for the default validation comparison. Contains original split assignments and validation references. |
| Your trained ASR `best.pt` | `ASR-cp/best.pt` | Already uploaded if you followed the earlier notebook. Reuse it. |

**Do not unzip manually.** The importer checks the archive and reads model/data files; bundled `code/` is not executed. The adapter ZIP already includes tokenizer files, so no separate full base-model download is needed; pinned base weights download automatically in Colab.

`resume_state.zip` is only needed for future full-state training resume, **not this inference workflow**. `reports.zip` alone does not contain model weights. If you only want inference without comparison, omit `reports.zip`, set `REPORTS_ZIP=''` and choose `MODE='baseline'` explicitly.

For the closest comparison to **56.7**, also ask your teammate for the **actual public ASR transcript JSONL used for that submission**, if available. Upload it to `MyDrive/Zalo-AI-Challenge-2026/inputs/teammate_public_asr.jsonl`; set `ASR_MODE='transcripts'` and `TRANSCRIPTS_PATH='/content/drive/MyDrive/Zalo-AI-Challenge-2026/inputs/teammate_public_asr.jsonl'` in Step 5. It must include final-request text and every user-history transcript. Otherwise this notebook decodes using your existing `best.pt`. Changing ASR can change the public score even with identical planner weights.

**Before Colab:** commit and push this notebook plus all new `training_pipeline/topic_*` files and the `training_pipeline/topic_agent/` directory. Select an **L4 or A100** runtime. Keep experiment names and Git revision fixed when resuming an interrupted inference run.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/Zalo-AI-Challenge-2026')
IMPORT_NAME = 'teammate_topic_20261109'
EXPERIMENT_NAME = 'topic_agent_compare_20261109_v2'  # New name if weights/code/settings change.
ADAPTER_ZIP = DRIVE_ROOT / 'imports' / IMPORT_NAME / 'best_adapter.zip'
REPORTS_ZIP = DRIVE_ROOT / 'imports' / IMPORT_NAME / 'reports.zip'
WORK_ROOT = '/content/lifemate_topic_work'
print('Adapter export:', ADAPTER_ZIP)
print('Validation reports:', REPORTS_ZIP)


## 2. Clone the implementation

Use a fresh checkout for this task. Existing checkouts are retained to avoid changing code mid-run. Set `REPO_REF` to your pushed commit SHA for reproducibility; changing code requires a new experiment name.


In [ ]:
import sys, subprocess, importlib
REPO_URL = 'https://github.com/votaquangnhat/LifeMate-Baseline.git'
REPO_REF = 'main'  # Prefer the pushed commit SHA.
PROJECT = Path('/content/lifemate-topic-agent')
if not PROJECT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
    subprocess.run(['git', '-C', str(PROJECT), 'checkout', REPO_REF], check=True)
revision = subprocess.check_output(['git','-C',str(PROJECT),'rev-parse','HEAD'],text=True).strip()
expected = subprocess.check_output(['git','-C',str(PROJECT),'rev-parse',REPO_REF],text=True).strip()
assert revision == expected, 'Existing checkout differs from REPO_REF. Select the intended commit explicitly.'
CODE_DIR = PROJECT / 'training_pipeline'
assert (CODE_DIR / 'topic_workflow.py').is_file(), 'Push the new topic workflow first; an old clone will not contain it.'
sys.path.insert(0, str(CODE_DIR))
for name in ('topic_workflow','topic_environment','topic_artifacts','colab_workflow'):
    sys.modules.pop(name, None)
importlib.invalidate_caches()
import topic_workflow
print('Repository commit:', revision)


## 3. Validate/import the selected model and prepare the runtime

The import remains separate from inference outputs. The source Kaggle package manifest is preserved; compatible Python packages and Unsloth commits are pinned from it while Colab's CUDA stack is retained. This is inference portability, not exact optimizer-state resume or a promise of bit-identical GPU results.

This cell prints the GPU name/memory. Models run in separate subprocesses, so the segmenter, compressor, BGE, ASR and planner do not occupy GPU memory together. Local model/dataset downloads and first-time history preparation can take a while; completed caches and sample results are reused.


In [ ]:
SESSION = topic_workflow.prepare(
    drive_root=DRIVE_ROOT, import_name=IMPORT_NAME, experiment_name=EXPERIMENT_NAME,
    adapter_zip=ADAPTER_ZIP, reports_zip=REPORTS_ZIP, work_root=WORK_ROOT,
)
# CPU contracts for imported-model loading, routing, thinking, and protected context.
subprocess.run([SESSION['python'], '-m', 'unittest', 'discover',
                '-s', str(CODE_DIR / 'topic_agent'), '-p', 'selftest_*.py'], check=True)


## 4. Compare agents before choosing one

| Setting | `baseline` | `enhanced` (experimental) |
|---|---|---|
| Adapter and action protocol | Teammate's exported model | Same model; no retraining |
| Total context / executor prompt ceiling | Saved settings (4096 / 2560 in supplied notebook) | 8192 / 7168 |
| History budget | Saved setting (1024) | 2048 |
| Retrieval | Teammate topical BM25+BGE | Adds final-request candidate retrieval; BGE ranks against that request |
| Context pressure | Teammate's original trimming | May remove descriptions/older history; never slices the final request, argument schemas or live tool results |

Run all three on the **same original validation IDs and reference states**. Clean transcripts are the default; optionally supply ASR transcripts for those exact validation IDs below. No public labels or held-out test scores select a variant. Higher local validation score wins; ties retain baseline. The validation set already selected the trained checkpoint, so this comparison is useful for choosing settings but is not an unbiased new test or a public-score prediction.

If protected context still cannot fit, enhanced mode records an early stop rather than feed a damaged request. More context may cost more GPU memory and time. Inspect the diagnostics as well as the score.

**Third variant — `read_guard`:** keeps the baseline settings and caches identical reads until any non-read call occurs. Only catalog-declared reads are cached; writes are never suppressed. Three repeated cached-read proposals end the loop with a diagnostic. Cached reads are not inserted into the official executed-call list. This is a separate ablation, not automatically combined with enhanced context.

This experiment was motivated by [Datyth's repo](https://github.com/Datyth/ZaloAI-Challenge-lifemate/tree/99c63e30398cc62ebff6e07bb3b1c9587172fb86). Its blanket deduplication rule was **not** copied: two local training examples correctly repeat identical writes. Its reported 58.02 uses a different non-thinking adapter; keep Thinking enabled for the imported teammate adapter.

**Optional ASR validation:** `VALIDATION_ASR_JSONL` must contain exactly the teammate's validation IDs, final text and all user-history transcripts. Do not supply public-test transcripts here. Use a separate experiment name when changing validation inputs. If the ASR trained on these validation recordings, this is an end-to-end diagnostic rather than a leakage-free held-out score. Leave it empty when that file is unavailable.


In [ ]:
VALIDATION_ASR_JSONL = ''  # Optional JSONL for EXACT original validation IDs; otherwise gold text.
COMPARISON = topic_workflow.compare_validation(SESSION, validation_transcripts=VALIDATION_ASR_JSONL)
print('Validation input:', COMPARISON['validation_input'])
print('Selected agent:', COMPARISON['chosen'])


## 5. Independent public inference — this cell can run by itself

After Step 4 completes once, this cell can run in a **fresh Colab runtime** without the earlier cells. `MODE='auto'` reads the saved comparison. If you skip validation, explicitly choose `baseline`, `enhanced`, or `read_guard`; there is no silent assumption that enhanced is better.

It restores the imported selected adapter and uses your trained ASR. It does not require `training_complete.json`, checkpoints or training cells. A Wi-Fi interruption can be resumed with the same paths, code and settings: completed simulated samples are reused. Partially completed samples restart from their initial simulated state; no external services are called.

Early stops are printed and logged. Complete, schema-valid submissions still export. GPU/runtime errors or incomplete output do not become a fabricated submission.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DRIVE_ROOT = Path('/content/drive/MyDrive/Zalo-AI-Challenge-2026')
IMPORT_NAME = 'teammate_topic_20261109'
EXPERIMENT_NAME = 'topic_agent_compare_20261109_v2'
ADAPTER_ZIP = DRIVE_ROOT / 'imports' / IMPORT_NAME / 'best_adapter.zip'
REPORTS_ZIP = DRIVE_ROOT / 'imports' / IMPORT_NAME / 'reports.zip'  # '' if not using validation.
WORK_ROOT = '/content/lifemate_topic_work'
MODE = 'auto'  # 'auto' after comparison, or 'baseline' / 'enhanced' / 'read_guard'.
ASR_MODE = 'gipformer'  # 'transcripts' to reuse an existing complete JSONL.
ASR_CHECKPOINT = str(DRIVE_ROOT / 'ASR-cp/best.pt')
ASR_DECODER = 'auto'
TRANSCRIPTS_PATH = ''  # In transcript mode, absolute path to your complete public_asr.jsonl.

import sys, subprocess, importlib
REPO_URL = 'https://github.com/votaquangnhat/LifeMate-Baseline.git'
REPO_REF = 'main'  # Prefer the pushed commit SHA.
PROJECT = Path('/content/lifemate-topic-agent')
if not PROJECT.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT)], check=True)
    subprocess.run(['git', '-C', str(PROJECT), 'checkout', REPO_REF], check=True)
revision = subprocess.check_output(['git','-C',str(PROJECT),'rev-parse','HEAD'],text=True).strip()
expected = subprocess.check_output(['git','-C',str(PROJECT),'rev-parse',REPO_REF],text=True).strip()
assert revision == expected, 'Existing checkout differs from REPO_REF. Select the intended commit explicitly.'
CODE_DIR = PROJECT / 'training_pipeline'
assert (CODE_DIR / 'topic_workflow.py').is_file(), 'Push the new topic workflow first; an old clone will not contain it.'
sys.path.insert(0, str(CODE_DIR))
for name in ('topic_workflow','topic_environment','topic_artifacts','colab_workflow'):
    sys.modules.pop(name, None)
importlib.invalidate_caches()
import topic_workflow
print('Repository commit:', revision)

SESSION = topic_workflow.prepare(
    drive_root=DRIVE_ROOT, import_name=IMPORT_NAME, experiment_name=EXPERIMENT_NAME,
    adapter_zip=ADAPTER_ZIP, reports_zip=REPORTS_ZIP, work_root=WORK_ROOT,
)

SUBMISSION = topic_workflow.public(
    SESSION, mode=MODE, asr_mode=ASR_MODE, asr_checkpoint=ASR_CHECKPOINT,
    asr_decoder=ASR_DECODER, transcripts_path=TRANSCRIPTS_PATH,
)
print('Ready to submit:', SUBMISSION)


## 6. Saved outputs and what to inspect

Everything persists under `MyDrive/Zalo-AI-Challenge-2026/`:

- `imports/teammate_topic_20261109/`: original ZIPs plus checked adapter/tokenizer/config metadata. No imported code runs.
- `runs/topic_agent_compare_20261109_v2/agent_comparison.json`: validation scores, selected mode and provenance.
- `runs/topic_agent_compare_20261109_v2/{baseline,enhanced,read_guard}/`: separate run configuration, validation metrics and execution traces.
- The selected mode's `public_test_best/submission.jsonl` and `predictions_with_trace.jsonl`: exact predictions and tool traces.
- `public_test_quality_report.json`: early stops, invalid actions, router fallback and context-trimming diagnostics.
- Root `submission.jsonl`: latest validated submission, also preserved in its run folder.

For further improvements, review validation `error_review.jsonl` to distinguish wrong app routing, missing history, incorrect arguments and premature `done`. Compare the same saved ASR transcripts when testing planner changes. Public traces cannot establish correctness without hidden labels. A future training run should use this adapter's router/open-app/tool-call protocol; do not resume it through the older compact-agent notebook.

Local CPU tests and real-tokenizer checks are provided. This notebook still needs your exported weights and Colab GPU run to establish runtime memory, validation gains and public score.

Each completed validation variant also writes `validation_error_analysis.json` and `validation_error_cases.jsonl`: state mismatch, no calls, missing/extra clarification, call-count discrepancies, and argument differences to inspect. These are diagnostic categories, not automatically proven causes. The original fuzzy score remains unchanged. Public output gets `public_error_diagnostics.json` with repeated calls and calls to apps absent from the training catalog; it does not invent correctness labels.

The external repo also reports regressions from synthetic-app SFT, global hotword biasing with its improved ASR, and added agent checks without reliable gains. None is enabled here based on that report alone. A future retraining experiment should test paired gold+ASR training examples and verified arithmetic/conditional reasoning targets using your original splits.


In [ ]:
from google.colab import files
files.download(str(SUBMISSION))
